In [5]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import glob
from xgrads import open_CtlDataset
from pathlib import Path
import netCDF4

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib.colors as mcolors

import ipywidgets as widgets
from IPython.display import display, clear_output

ncl_cmap = LinearSegmentedColormap.from_list(
    "BlueWhiteOrangeRed",
    ["#2166ac", "#67a9cf", "#ffffff", "#fdae61", "#b2182b"],
    N=256
)


plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_colwidth", 160)

print("Python OK")

Python OK


In [6]:
BASE_DIR = Path.cwd()

NEW_DIR = Path.cwd() / ".." / ".." / ".." / "SPEEDY_access" / "output" / "exp_102"
OLD_DIR = Path.cwd() / ".." / ".." / ".." / "SPEEDY_access" / "output" / "exp_101"

print("NEW_DIR:", NEW_DIR, NEW_DIR.exists())
print("OLD_DIR:", OLD_DIR, OLD_DIR.exists())

# SPEEDY precipitation components
SPEEDY_VARIABLES = ["PRECLS", "PRECNV","SNOW"]
ACCESS_VARIABLES = ["prra","prsn"]

# Output directory
PRRA_OUT_DIR = (Path.cwd() / ".." / ".." / "access_forcing").resolve()
PRRA_OUT_DIR.mkdir(parents=True, exist_ok=True)

WRITE_ONE_FILE_PER_YEAR = True

# Keep SPEEDY grid unchanged for now
SHIFT_LONGITUDE_TO_MINUS180_180 = False
SORT_LATITUDE_NORTH_TO_SOUTH = False

print("Output directory:", PRRA_OUT_DIR)

NEW_DIR: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/scripts/access_forcing/../../../SPEEDY_access/output/exp_102 True
OLD_DIR: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/scripts/access_forcing/../../../SPEEDY_access/output/exp_101 True
Output directory: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing


In [7]:
for ctl in Path(NEW_DIR).glob("attm102.ctl"):
    print("=" * 80)
    print(ctl.name)

    ds = open_CtlDataset(str(ctl))
ds

attm102.ctl


<xarray.Dataset> Size: 17GB
Dimensions:  (time: 8760, lev: 8, lat: 48, lon: 96)
Coordinates:
  * time     (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lev      (lev) float64 64B 925.0 850.0 700.0 500.0 300.0 200.0 100.0 30.0
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Data variables: (12/43)
    GH       (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    TEMP     (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    U        (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    V        (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    Q        (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    RH       (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    ...       ...
    SHF      (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    LSHF     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SSHF     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SSRD     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SLRD     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SNOW     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
Attributes:
    comment:  geopotential height               [m]
    storage:  99
    title:    Means/variances
    undef:    9.999e+19
    pdef:     None

In [8]:
# Inspect source precipitation

for var_name in SPEEDY_VARIABLES:
    if var_name not in ds:
        raise KeyError(f"{var_name!r} is absent from SPEEDY dataset. Available: {list(ds.data_vars)}")

    var = ds[var_name]

    print(f"\n{'='*60}\n{var_name}\n{'='*60}")
    print(var)
    print("Dimensions:", var.dims)
    print("Shape:", var.shape)
    print("Dtype:", var.dtype)
    print("Attributes:", var.attrs)
    print("Range:", float(var.min().compute()), "to", float(var.max().compute()))
    print("Global mean:", float(var.mean(skipna=True).compute()))

# Total precipitation
#prec = ds["PRECLS"] + ds["PRECNV"] - ds["SNOW"]

#print("Total precipitation range:", float(prec.min().compute()), "to", float(prec.max().compute()))
#print("Total precipitation global mean:", float(prec.mean(skipna=True).compute()))

# Temporal resolution
#dt_hours = np.diff(ds.time.values) / np.timedelta64(1, "h")

#print("\nUnique output intervals [hours]:", np.unique(dt_hours))
#print("First time:", ds.time.values[0])
#print("Last time:", ds.time.values[-1])


PRECLS
<xarray.DataArray 'PRECLS' (time: 8760, lat: 48, lon: 96)> Size: 161MB
dask.array<reshape, shape=(8760, 48, 96), dtype=>f4, chunksize=(1, 48, 96), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Attributes:
    comment:  large-scale precipitation    [mm/day]
    storage:  99
Dimensions: ('time', 'lat', 'lon')
Shape: (8760, 48, 96)
Dtype: >f4
Attributes: {'comment': 'large-scale precipitation    [mm/day]', 'storage': '99'}
Range: 0.0 to 62.547828674316406
Global mean: 0.9503865838050842

PRECNV
<xarray.DataArray 'PRECNV' (time: 8760, lat: 48, lon: 96)> Size: 161MB
dask.array<reshape, shape=(8760, 48, 96), dtype=>f4, chunksize=(1, 48, 96), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lat      (

In [9]:
# =========================
# Build prra and prsn

# SPEEDY precipitation diagnostics are in mm/day water equivalent.
# prra = liquid precipitation, prsn = snowfall.
prra = ((ds["PRECLS"] + ds["PRECNV"] - ds["SNOW"]) / 86400.0).clip(min=0.0)
prsn = (ds["SNOW"] / 86400.0).clip(min=0.0)

prra = prra.rename("prra").astype("float32")
prsn = prsn.rename("prsn").astype("float32")

if SHIFT_LONGITUDE_TO_MINUS180_180:
    prra = prra.assign_coords(lon=((prra.lon + 180.0) % 360.0) - 180.0).sortby("lon")
    prsn = prsn.assign_coords(lon=((prsn.lon + 180.0) % 360.0) - 180.0).sortby("lon")

if SORT_LATITUDE_NORTH_TO_SOUTH:
    prra = prra.sortby("lat", ascending=False)
    prsn = prsn.sortby("lat", ascending=False)

prra.attrs = {
    "standard_name": "rainfall_flux",
    "long_name": "Rainfall Flux",
    "units": "kg m-2 s-1",
    "cell_methods": "area: time: mean",
    "source_variable": "PRECLS + PRECNV - SNOW",
    "source_model": "SPEEDY",
    "mapping_note": "SPEEDY liquid precipitation converted from mm day-1 to kg m-2 s-1",
}

prsn.attrs = {
    "standard_name": "snowfall_flux",
    "long_name": "Snowfall Flux",
    "units": "kg m-2 s-1",
    "cell_methods": "area: time: mean",
    "source_variable": "SNOW",
    "source_model": "SPEEDY",
    "mapping_note": "SPEEDY snowfall converted from mm day-1 water equivalent to kg m-2 s-1",
}

prra_ds = prra.to_dataset()
prsn_ds = prsn.to_dataset()

# JRA55-do 3hr convention: 3-hour mean, time at interval midpoint
raw_time = prra_ds.time.copy()
new_time = raw_time + np.timedelta64(90, "m")

prra_ds = prra_ds.assign_coords(time=new_time)
prsn_ds = prsn_ds.assign_coords(time=new_time)

time_bnds = np.stack([
    raw_time.values,
    (raw_time + np.timedelta64(3, "h")).values
], axis=1)

prra_ds["time_bnds"] = xr.DataArray(
    time_bnds, dims=("time", "bnds"),
    coords={"time": prra_ds.time, "bnds": [0, 1]}
)

prsn_ds["time_bnds"] = xr.DataArray(
    time_bnds, dims=("time", "bnds"),
    coords={"time": prsn_ds.time, "bnds": [0, 1]}
)

for forcing_ds in [prra_ds, prsn_ds]:
    forcing_ds["lat"].attrs.update({"standard_name": "latitude", "long_name": "Latitude", "units": "degrees_north", "axis": "Y"})
    forcing_ds["lon"].attrs.update({"standard_name": "longitude", "long_name": "Longitude", "units": "degrees_east", "axis": "X"})
    forcing_ds["time"].attrs.update({"standard_name": "time", "long_name": "time", "axis": "T", "bounds": "time_bnds"})

prra_ds.attrs = {
    "Conventions": "CF-1.7",
    "title": "SPEEDY forcing for ACCESS-OM2",
    "source": "SPEEDY model output",
    "frequency": "3hr",
    "history": "Created from SPEEDY PRECLS, PRECNV and SNOW and exported as prra",
    "comment": "3-hourly mean liquid precipitation forcing following JRA55-do temporal convention.",
}

prsn_ds.attrs = {
    "Conventions": "CF-1.7",
    "title": "SPEEDY forcing for ACCESS-OM2",
    "source": "SPEEDY model output",
    "frequency": "3hr",
    "history": "Created from SPEEDY SNOW and exported as prsn",
    "comment": "3-hourly mean snowfall forcing following JRA55-do temporal convention.",
}

prra_ds, prsn_ds

(<xarray.Dataset> Size: 162MB
 Dimensions:    (lat: 48, lon: 96, time: 8760, bnds: 2)
 Coordinates:
   * lat        (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
   * lon        (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
   * time       (time) datetime64[ns] 70kB 1989-01-01T01:30:00 ... 1991-12-31T...
   * bnds       (bnds) int64 16B 0 1
 Data variables:
     prra       (time, lat, lon) float32 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
     time_bnds  (time, bnds) datetime64[ns] 140kB 1989-01-01 ... 1992-01-01
 Attributes:
     Conventions:  CF-1.7
     title:        SPEEDY forcing for ACCESS-OM2
     source:       SPEEDY model output
     frequency:    3hr
     history:      Created from SPEEDY PRECLS, PRECNV and SNOW and exported as...
     comment:      3-hourly mean liquid precipitation forcing following JRA55-...,
 <xarray.Dataset> Size: 162MB
 Dimensions:    (lat: 48, lon: 96, time: 8760, bnds: 2)
 Coordinates:
   * lat       

In [11]:
# NetCDF encoding

field_encoding = {
    "dtype": "float32", "zlib": True, "complevel": 4, "shuffle": True,
    "_FillValue": np.float32(1.0e20),
    "chunksizes": (1, prra.sizes["lat"], prra.sizes["lon"]),
}

encoding = {
    "prra": field_encoding,
    "time": {"dtype": "float64", "units": "days since 1900-01-01 00:00:00", "calendar": "gregorian", "_FillValue": None},
    "time_bnds": {"dtype": "float64", "units": "days since 1900-01-01 00:00:00", "calendar": "gregorian", "_FillValue": None},
    "lat": {"dtype": "float64", "_FillValue": None},
    "lon": {"dtype": "float64", "_FillValue": None},
}

field_encoding_s = {
    "dtype": "float32", "zlib": True, "complevel": 4, "shuffle": True,
    "_FillValue": np.float32(1.0e20),
    "chunksizes": (1, prsn.sizes["lat"], prsn.sizes["lon"]),
}

encoding_s = {
    "prsn": field_encoding_s,
    "time": {"dtype": "float64", "units": "days since 1900-01-01 00:00:00", "calendar": "gregorian", "_FillValue": None},
    "time_bnds": {"dtype": "float64", "units": "days since 1900-01-01 00:00:00", "calendar": "gregorian", "_FillValue": None},
    "lat": {"dtype": "float64", "_FillValue": None},
    "lon": {"dtype": "float64", "_FillValue": None},
}

print(encoding.keys())
print(encoding_s.keys())

dict_keys(['prra', 'time', 'time_bnds', 'lat', 'lon'])
dict_keys(['prsn', 'time', 'time_bnds', 'lat', 'lon'])


In [13]:
# =========================
# Write NetCDF files

written_files = []

if WRITE_ONE_FILE_PER_YEAR:
    years = np.unique(prra_ds.time.dt.year.values)

    for year in years:
        yearly_prra = prra_ds.sel(time=str(int(year)))
        yearly_prsn = prsn_ds.sel(time=str(int(year)))

        if yearly_prra.sizes["time"] != 2920:
            raise ValueError(f"{year}: prra expected 2920 records, got {yearly_prra.sizes['time']}")

        if yearly_prsn.sizes["time"] != 2920:
            raise ValueError(f"{year}: prsn expected 2920 records, got {yearly_prsn.sizes['time']}")

        # Rainfall
        output_file = PRRA_OUT_DIR / f"prra_SPEEDY_{int(year)}.nc"

        yearly_prra.to_netcdf(
            output_file,
            mode="w",
            format="NETCDF4",
            engine="netcdf4",
            unlimited_dims=["time"],
            encoding=encoding,
        )

        written_files.append(output_file)
        print(f"Wrote {output_file.name}: {yearly_prra.sizes['time']} records, {output_file.stat().st_size / 1024**2:.2f} MB")

        # Snowfall
        output_file = PRRA_OUT_DIR / f"prsn_SPEEDY_{int(year)}.nc"

        yearly_prsn.to_netcdf(
            output_file,
            mode="w",
            format="NETCDF4",
            engine="netcdf4",
            unlimited_dims=["time"],
            encoding=encoding_s,
        )

        written_files.append(output_file)
        print(f"Wrote {output_file.name}: {yearly_prsn.sizes['time']} records, {output_file.stat().st_size / 1024**2:.2f} MB")

else:
    # Rainfall
    output_file = PRRA_OUT_DIR / "prra_SPEEDY_all_years.nc"

    prra_ds.to_netcdf(
        output_file,
        mode="w",
        format="NETCDF4",
        engine="netcdf4",
        unlimited_dims=["time"],
        encoding=encoding,
    )

    written_files.append(output_file)
    print(f"Wrote {output_file.name}: {prra_ds.sizes['time']} records, {output_file.stat().st_size / 1024**2:.2f} MB")

    # Snowfall
    output_file = PRRA_OUT_DIR / "prsn_SPEEDY_all_years.nc"

    prsn_ds.to_netcdf(
        output_file,
        mode="w",
        format="NETCDF4",
        engine="netcdf4",
        unlimited_dims=["time"],
        encoding=encoding_s,
    )

    written_files.append(output_file)
    print(f"Wrote {output_file.name}: {prsn_ds.sizes['time']} records, {output_file.stat().st_size / 1024**2:.2f} MB")

written_files

Wrote prra_SPEEDY_1989.nc: 2920 records, 23.02 MB
Wrote prsn_SPEEDY_1989.nc: 2920 records, 10.43 MB
Wrote prra_SPEEDY_1990.nc: 2920 records, 22.85 MB
Wrote prsn_SPEEDY_1990.nc: 2920 records, 10.09 MB
Wrote prra_SPEEDY_1991.nc: 2920 records, 22.71 MB
Wrote prsn_SPEEDY_1991.nc: 2920 records, 10.06 MB


[PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/prra_SPEEDY_1989.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/prsn_SPEEDY_1989.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/prra_SPEEDY_1990.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/prsn_SPEEDY_1990.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/prra_SPEEDY_1991.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/prsn_SPEEDY_1991.nc')]

In [ ]:
# Reopen and verify output

if not written_files:
    raise RuntimeError("No NetCDF files were written")

check_file = written_files[0]

with xr.open_dataset(check_file, decode_times=True) as check:
    print(check)
    print("\nVariable attributes:", check[ACCESS_VARIABLE].attrs)
    print("\nEncoding:", check[ACCESS_VARIABLE].encoding)
    print("\nTime:", check.time.values[0], "to", check.time.values[-1])

    if check.sizes["time"] != 2920:
        raise ValueError(f"Expected 2920 records, got {check.sizes['time']}")

    if check.attrs.get("frequency") != "3hrPt":
        raise ValueError(f"Unexpected frequency: {check.attrs.get('frequency')}")

    if "time_bnds" not in check:
        raise ValueError("time_bnds is missing")

    dt_hours = np.diff(check.time.values) / np.timedelta64(1, "h")
    width_hours = (check.time_bnds[:, 1] - check.time_bnds[:, 0]).values / np.timedelta64(1, "h")
    midpoints = check.time_bnds[:, 0].values + (check.time_bnds[:, 1].values - check.time_bnds[:, 0].values) / 2

    if not np.all(dt_hours == 3):
        raise ValueError(f"Unexpected time intervals: {np.unique(dt_hours)} h")
    if not np.all(width_hours == 3):
        raise ValueError(f"Unexpected time_bnds widths: {np.unique(width_hours)} h")
    if not np.array_equal(midpoints, check.time.values):
        raise ValueError("time is not the midpoint of time_bnds")

    if check.time.encoding.get("units") != "days since 1900-01-01":
        raise ValueError(f"Unexpected time units: {check.time.encoding.get('units')}")
    if check.time.encoding.get("calendar") != "gregorian":
        raise ValueError(f"Unexpected calendar: {check.time.encoding.get('calendar')}")

    tas_min = float(check[ACCESS_VARIABLE].min())
    tas_max = float(check[ACCESS_VARIABLE].max())
    print(f"\nRange [K]: {tas_min:.3f} to {tas_max:.3f}")

    source_first = tas.sel(time=check.time.values[0]).compute()
    output_first = check[ACCESS_VARIABLE].isel(time=0).load()
    max_abs_difference = float(np.abs(source_first - output_first).max())

    print("Maximum absolute difference after NetCDF round trip:", max_abs_difference)

    if max_abs_difference != 0.0:
        raise ValueError("NetCDF round trip changed tas values")

print("Output verification passed")